In [1]:
import pandas as pd
from pathlib import Path

def file_metadata(path: Path):
    return {
        "filename": path.name,
        "size_gb": round(path.stat().st_size / 1e9, 2),
        "modified": pd.to_datetime(path.stat().st_mtime, unit="s"),
    }


In [2]:
df = pd.DataFrame(columns=["filename", "size_gb", "modified"])


In [ ]:
import ipywidgets as widgets

path_input = widgets.Text(
    description="Path:",
    placeholder="/path/to/file_or_directory",
    layout=widgets.Layout(width="600px")
)

add_button = widgets.Button(
    description="ADD",
    button_style="success",
    layout=widgets.Layout(width="120px")
)

clear_button = widgets.Button(
    description="Clear",
    button_style="danger",
    layout=widgets.Layout(width="120px")
)

add_output = widgets.Output()


In [ ]:
# def on_add_clicked(b):
#     global df
#     add_output.clear_output()
#     with add_output:
#         p = Path(path_input.value).expanduser()

#         if not p.exists():
#             print("Path does not exist:", p)
#             return

#         global df

#         # Case 1: user selects a directory → add all files inside
#         if p.is_dir():
#             files = list(p.glob("*"))
#             if not files:
#                 print("Directory is empty:", p)
#                 return

#             new_rows = [file_metadata(f) for f in files if f.is_file()]
#             df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
#             print(f"Added {len(new_rows)} files from directory:", p)

#         # Case 2: user selects a single file
#         elif p.is_file():
#             df = pd.concat([df, pd.DataFrame([file_metadata(p)])], ignore_index=True)
#             print("Added file:", p.name)

#         else:
#             print("Path is neither file nor directory:", p)

In [ ]:
# add_button.on_click(on_add_clicked)


In [6]:
def df_to_html(df):
    return df.to_html(index=False)

table = widgets.HTML(
    value=df_to_html(df),
    layout=widgets.Layout(border="1px solid #ccc", height="300px", overflow="auto")
)


In [ ]:
def refresh_table():
    table.value = df_to_html(df)
    refresh_selector()


In [ ]:
def on_clear_clicked(b):
    global df
    df = pd.DataFrame(columns=["filename", "size_gb", "modified"])
    refresh_table()

In [ ]:
clear_button.on_click(on_clear_clicked)

In [ ]:
def on_add_clicked(b):
    add_output.clear_output()
    with add_output:
        p = Path(path_input.value).expanduser()

        if not p.exists():
            print("Path does not exist:", p)
            return

        global df

        # Helper: check if filename already exists in df
        def already_in_df(path):
            return path.name in df["filename"].values

        # Directory → add all files
        if p.is_dir():
            files = [f for f in p.glob("*") if f.is_file()]
            new_rows = []

            for f in files:
                if already_in_df(f):
                    print(f"Skipping (already in table): {f.name}")
                else:
                    new_rows.append(file_metadata(f))

            if new_rows:
                df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
                print(f"Added {len(new_rows)} new files from directory:", p)
            else:
                print("No new files to add from directory.")

        # Single file → add only if not already present
        elif p.is_file():
            if already_in_df(p):
                print("File already in table:", p.name)
            else:
                df = pd.concat([df, pd.DataFrame([file_metadata(p)])], ignore_index=True)
                print("Added file:", p.name)

        else:
            print("Path is neither file nor directory:", p)
            return

        refresh_table()

add_button.on_click(on_add_clicked)

In [ ]:
file_selector = widgets.SelectMultiple(
    options=df["filename"].tolist(),
    description="Files:",
    layout=widgets.Layout(width="400px", height="200px")
)


In [ ]:
def refresh_selector():
    file_selector.options = df["filename"].tolist()

# )

SyntaxError: unmatched ')' (363627906.py, line 9)

In [ ]:
load_button = widgets.Button(
    description="Load",
    button_style="info",
    layout=widgets.Layout(width="120px")
)

load_output = widgets.Output()

In [ ]:
def on_load_clicked(b):
    load_output.clear_output()
    with load_output:
        selected = list(file_selector.value)
        if not selected:
            print("No files selected.")
            return

        print("Files selected for loading:")
        for fname in selected:
            print(" -", fname)

        # Here you can call your real loading function
        # load_heavy_data(fname)


In [ ]:
load_button.on_click(on_load_clicked)

In [ ]:
widgets.VBox([
    widgets.HBox([path_input, add_button, clear_button]),
    add_output,
    table,
    widgets.HBox([file_selector, load_button]),
    load_output
])